# 🏛️ Observatorio Ciudadano de Contratación Pública (SECOP II)
### Asistente Híbrido en la Nube (RAG + SQL) y Modelo de Machine Learning Clásico para Estimación de Riesgo de Prórrogas

---

## 📋 Descripción General del Proyecto
Este notebook constituye la base de desarrollo del **Observatorio Ciudadano de Contratación Pública** sobre datos abiertos de **SECOP II** (Colombia Compra Eficiente). El proyecto aborda dos problemas complementarios para veedurías ciudadanas, periodistas de datos y equipos de control interno:

1. **Componente A — Asistente en la Nube (RAG + SQL):** Combina consultas analíticas sobre una base de datos relacional (`SQLite`) con recuperación semántica sobre normativa de contratación estatal (`Qdrant Cloud` + `Gemini Embedding 2`), respondiendo en lenguaje natural con fuentes verificables.
2. **Componente B — Machine Learning Clásico:** Estima la probabilidad de que un contrato público requiera **prórroga de plazo** (`dias_adicionados > 0`) utilizando exclusivamente variables conocidas al momento de la firma (previniendo fuga de información y validando con partición temporal).

> ⚠️ **Aviso de Uso Responsable:** Los resultados de este proyecto describen patrones estadísticos e informativos y **no constituyen evidencia de irregularidades ni reemplazan una auditoría oficial**. Toda alerta o hallazgo debe verificarse directamente en el expediente electrónico original a través del campo `urlproceso`.

> 📜 **Licencia y Atribución de Datos (CC BY-SA 4.0):** Datos originarios de *«SECOP II – Contratos Electrónicos»*, Colombia Compra Eficiente, Portal Nacional de Datos Abiertos (`datos.gov.co`). Los datos derivados se comparten bajo licencia **CC BY-SA 4.0**. Este proyecto es independiente y no está afiliado ni respaldado por Colombia Compra Eficiente.

--- 
## ⚙️ 0. Configuración del Entorno, Librerías y Variables de Entorno
**Objetivo de la sección:**
- Importar las librerías necesarias para ingesta de datos (`requests`, `pandas`, `sqlite3`), procesamiento documental y vectorización (`langchain`, `qdrant-client`, `google-genai`) y modelado (`scikit-learn`, `joblib`).
- Cargar las credenciales de forma segura desde el archivo `.env` (`QDRANT_URL`, `QDRANT_API_KEY`, `GEMINI_API_KEY`, `SOCRATA_APP_TOKEN`) sin exponer ninguna clave en el código ni en las salidas del notebook.
- Fijar semillas de aleatoriedad (`RANDOM_SEED = 42`) para garantizar reproducibilidad.

In [ ]:
# TODO: Importación de librerías estándar, manejo de rutas y carga de variables de entorno (.env)
# TODO: Definición de constantes globales (RANDOM_SEED, rutas de almacenamiento local, fecha de consulta)

---
# 📦 ETAPA 1: Ingesta, Limpieza y Base de Datos Compartida (SECOP II)

## 1.1 Extracción Reproducible vía API Socrata (`datos.gov.co`)
**Objetivo de la sección (RF-2 / RF-ML1):**
- Consultar el endpoint oficial `https://www.datos.gov.co/resource/jbjy-vk9h.json` filtrando por un **departamento específico** y un **rango de 1 a 2 años** (objetivo: entre 20.000 y 100.000 contratos).
- Implementar paginación mediante `$limit` y `$offset` ordenando por `id_contrato`.
- Registrar de forma explícita la **fecha exacta de consulta** para garantizar trazabilidad frente a la actualización diaria de SECOP II.

In [ ]:
# TODO: Definir parámetros de consulta SoQL ($select, $where, $order, $limit, $offset)
# TODO: Ejecutar descarga paginada desde la API de Datos Abiertos y registrar fecha de corte

## 1.2 Limpieza General, Privacidad y Diccionario de Columnas
**Objetivo de la sección:**
- **Protección de datos personales (Ley 1581 de 2012):** Excluir cualquier columna con nombres, documentos o domicilios de representantes legales, supervisores u ordenadores del gasto, así como datos bancarios. Filtrar o anonimizar contratistas personas naturales en vistas públicas.
- Eliminar duplicados por llave única (`id_contrato`).
- Estandarizar formatos de fecha (`fecha_de_firma`, etc.) y tipos numéricos (`valor_del_contrato`, `dias_adicionados`).
- Documentar el diccionario de las columnas seleccionadas con base en la Ficha Técnica de Colombia Compra Eficiente.

In [ ]:
# TODO: Filtrado de columnas permitidas y exclusión estricta de datos personales y bancarios
# TODO: Conversión de tipos de datos, normalización de textos y eliminación de duplicados por id_contrato

## 1.3 Carga en SQLite e Indexación
**Objetivo de la sección (RF-2):**
- Almacenar el subconjunto limpio en una base de datos local `SQLite` (tabla `contratos`).
- Crear índices sobre las columnas de filtrado frecuente (`departamento`, `modalidad_de_contratacion`, `tipo_de_contrato`, `fecha_de_firma`, `nombre_entidad`).
- Verificar que el conteo de registros en `SQLite` coincida exactamente con el subconjunto descargado.

In [ ]:
# TODO: Crear conexión SQLite, volcar tabla 'contratos' y crear índices de búsqueda
# TODO: Ejecutar consulta de verificación de conteo e integridad

---
# 🧠 ETAPA 2: Componente A — Asistente en la Nube (RAG Normativo + SQL)

## 2.1 Preparación del Corpus Normativo y Metadatos
**Objetivo de la sección (RF-3):**
- Cargar los textos normativos en formato Markdown:
  1. **Ley 80 de 1993** (Estatuto General de Contratación de la Administración Pública).
  2. **Ley 1150 de 2007** (Eficiencia y transparencia; modalidades de selección).
  3. **Tercer documento:** Título de Contratación Estatal del **Decreto 1082 de 2015** o **Manual de Datos Abiertos del SECOP**.
- Estructurar los metadatos obligatorios de cada documento: `documento`, `tipo_fuente`, `tema`, `fecha` y `articulo`.

In [ ]:
# TODO: Lectura de archivos Markdown del corpus normativo y extracción/asignación de metadatos

## 2.2 Estrategia de Fragmentación (Chunking) y Comparación Experimental
**Objetivo de la sección (RF-4):**
- Fragmentar el corpus respetando la estructura por artículo mediante `RecursiveCharacterTextSplitter`.
- Comparar **al menos tres configuraciones de tamaño (`chunk_size` y `chunk_overlap`)**.
- Generar una tabla comparativa con: número total de fragmentos, longitud media, mínima y máxima, verificando que ningún fragmento quede cortado a mitad de palabra.
- Justificar la elección de la configuración óptima.

In [ ]:
# TODO: Configurar al menos 3 tamaños en RecursiveCharacterTextSplitter
# TODO: Construir tabla comparativa de métricas de chunking y seleccionar el tamaño definitivo

## 2.3 Aprovisionamiento en Qdrant Cloud y Vectorización por Lotes
**Objetivo de la sección (RF-1, RF-5):**
- Conectar al clúster gratuito en **Qdrant Cloud** y crear la colección `normativa_contratacion` con métrica de **distancia coseno** y dimensión **768** (`Gemini Embedding 2`).
- Aplicar los prefijos requeridos por el modelo de embeddings:
  - Para documentos: `title: <documento - artículo> | text: <contenido>`
  - Para consultas: `task: search result | query: <pregunta>`
- Vectorizar por lotes respetando las cuotas gratuitas de Google AI Studio, respaldar los vectores en disco y realizar `upsert` por lotes en Qdrant.
- Confirmar que el total de puntos en Qdrant sea igual al número de fragmentos generados.

In [ ]:
# TODO: Inicializar cliente Qdrant Cloud y crear colección con VectorParams(size=768, distance=Distance.COSINE)
# TODO: Generar embeddings por lotes con Gemini Embedding 2, guardar respaldo local y hacer upsert en Qdrant

## 2.4 Función de Recuperación Semántica y Consultas SQL Seguras
**Objetivo de la sección (RF-6, RF-8):**
- Implementar la función de búsqueda vectorial `buscar_normativa(pregunta, top_k, filtro_documento=None)` que retorne una tabla con: `texto`, `score_coseno`, `documento` y `articulo`.
- Implementar el generador/ejecutor de consultas SQL en **modo solo lectura**, garantizando que solo se ejecute una única sentencia `SELECT` con `LIMIT` de seguridad y haciendo visible el SQL generado.

In [ ]:
# TODO: Definir función de recuperación semántica Top-k sobre Qdrant con filtro opcional por documento
# TODO: Definir herramienta de consulta SQL de solo lectura (validación de SELECT único y límite de filas)

## 2.5 Evaluación del Sistema RAG con Banco de Preguntas (10–15 preguntas)
**Objetivo de la sección (RF-7):**
- Evaluar el sistema con un banco de **10 a 15 preguntas** distribuidas en cuatro categorías:
  1. **Directas** (ej. *¿Cuándo procede la contratación directa?*)
  2. **Parafraseadas** (ej. *¿En qué casos una entidad puede contratar sin licitación?*)
  3. **Consultas SQL y Mixtas** (cifras de contratos + marco legal en Ley 1150/Ley 80)
  4. **Sin respuesta en el corpus** (control de alucinaciones, ej. *¿Cuál fue el contrato más barato firmado en Francia?*)
- Calcular métricas de validación: score coseno promedio, mínimo y máximo, **Hit Rate @ Top-1 y Top-3**, latencia por consulta (objetivo `< 5s`) y análisis de fallos.

In [ ]:
# TODO: Definir el banco de 10-15 preguntas de validación con su fuente esperada
# TODO: Ejecutar evaluación automatizada, calcular métricas (Top-1, Top-3, scores, latencia) y documentar análisis de fallos

---
# 📈 ETAPA 3: Componente B — Machine Learning Clásico (Riesgo de Prórroga)

## 3.1 Definición de la Variable Objetivo y Análisis Exploratorio de Datos (EDA)
**Objetivo de la sección (Paso 1 ML):**
- Definir la variable objetivo binaria: `prorroga = 1` si `dias_adicionados > 0`, de lo contrario `0`.
- **Validación previa:** Analizar el porcentaje de nulos y la proporción de desbalance de clases. (Si se requiere contraste adicional, cruzar con el dataset de Adiciones o documentar la variante elegida).
- Explorar distribuciones por modalidad de contratación, tipo de contrato, sector, departamento y relación entre `log(valor_del_contrato)` y la presencia de prórrogas.

In [ ]:
# TODO: Construcción y auditoría de la variable objetivo (dias_adicionados > 0)
# TODO: Gráficos exploratorios (distribución de clases, nulos, tasas de prórroga por modalidad/tipo y valor en escala log)

## 3.2 Selección de Variables y Prevención de Fuga de Información (Data Leakage)
**Objetivo de la sección (Sección 5.2 / RF-ML2):**
- **Variables permitidas (conocidas al firmar):** `tipo_de_contrato`, `modalidad_de_contratacion`, `sector`, `rama`, `orden`, `departamento`, `es_pyme`, `es_grupo`, `habilita_pago_adelantado`, `origen_de_los_recursos`, `codigo_de_categoria_principal` (segmento UNSPSC), `valor_del_contrato` (escala logarítmica), mes/año de firma y duración planeada inicial.
- **Variables excluidas por fuga de información:** `dias_adicionados` (es el target), `valor_pagado`, `valor_facturado`, `valor_pendiente_de_pago`, `estado_contrato`, `liquidaci_n`, fechas de liquidación y `reversion`.
- **Auditoría temporal de columnas:** Verificar que `valor_del_contrato` y la duración inicial no incluyan modificaciones posteriores a la firma.

In [ ]:
# TODO: Definir listas explícitas de FEATURES_NUMERICAS y FEATURES_CATEGORICAS conocidas al momento de la firma
# TODO: Auditar ausencia de variables post-firma (data leakage)

## 3.3 División Temporal y Pipeline de Preprocesamiento en Scikit-Learn
**Objetivo de la sección (RF-ML2):**
- Realizar **división temporal estricta**: entrenar con el periodo histórico (ej. año anterior / primeros meses) y evaluar sobre el periodo más reciente.
- Construir un `ColumnTransformer` y `Pipeline` en `scikit-learn` que ejecute:
  - Agrupación de categorías poco frecuentes (`min_frequency`).
  - Imputación de valores faltantes (`SimpleImputer`).
  - Codificación categórica (`OneHotEncoder(handle_unknown='ignore')`).
  - Escalado numérico (`StandardScaler`) cuando el algoritmo lo requiera.
- Ajustar (`fit`) todas las transformaciones **únicamente sobre el conjunto de entrenamiento**.

In [ ]:
# TODO: Partición temporal (Train / Test split basado en fecha_de_firma)
# TODO: Definición del ColumnTransformer para variables numéricas y categóricas

## 3.4 Entrenamiento y Comparación de Modelos Clásicos
**Objetivo de la sección (RF-ML3):**
- Entrenar y comparar mediante validación cruzada sobre el conjunto de entrenamiento:
  1. **Baseline:** `DummyClassifier` (referencia mínima).
  2. **Modelo Lineal:** `LogisticRegression` (con `class_weight='balanced'`).
  3. **Ensamble de Árboles (Bagging):** `RandomForestClassifier` (con manejo de desbalance).
  4. **Ensamble por Boosting:** `HistGradientBoostingClassifier` o `GradientBoostingClassifier`.
- Realizar ajuste de hiperparámetros exclusivamente sobre el conjunto de entrenamiento.

In [ ]:
# TODO: Configurar pipelines para Baseline, Regresión Logística, Random Forest y Gradient Boosting
# TODO: Entrenamiento, validación cruzada y búsqueda de hiperparámetros en el conjunto de entrenamiento

## 3.5 Evaluación en Prueba Temporal, Interpretabilidad y Análisis de Errores
**Objetivo de la sección (RF-ML4):**
- Evaluar todos los modelos sobre el **conjunto de prueba temporal** reportando:
  - `Precision`, `Recall` y `F1-Score` de la clase positiva (`prorroga = 1`).
  - `ROC-AUC` y especialmente `PR-AUC` (Average Precision), clave en escenarios desbalanceados.
  - Matriz de confusión comparativa frente al baseline.
- Calcular la **importancia de variables por permutación (`permutation_importance`)** y traducir los hallazgos a lenguaje claro para usuarios no técnicos.
- Realizar un **análisis de errores** desglosado por modalidad de contratación y departamento.

In [ ]:
# TODO: Generar tabla comparativa de métricas en el conjunto de prueba temporal y graficar Matrices de Confusión / Curvas PR
# TODO: Calcular importancia de variables por permutación y analizar falsos positivos / falsos negativos por modalidad

---
# 🚀 ETAPA 4: Exportación de Artefactos, Verificación y Ficha del Modelo (Model Card)

## 4.1 Serialización del Pipeline y Prueba de Consistencia para Streamlit
**Objetivo de la sección (RF-ML6):**
- Exportar el `Pipeline` completo ganador en formato `.joblib` junto con los metadatos necesarios para la interfaz (lista de valores permitidos para los selectores, umbrales de riesgo y métricas de validación).
- Ejecutar una prueba de inferencia individual cargando el archivo `.joblib` desde disco y comprobando que la predicción coincide exactamente con la del notebook y toma `< 2 segundos`.

In [ ]:
# TODO: Guardar artefacto del modelo con joblib.dump() y metadatos auxiliares en formato JSON
# TODO: Cargar el artefacto y verificar predicción unitaria (tiempo de respuesta y coincidencia exacta)

## 4.2 Ficha del Modelo (Model Card) y Conclusiones del Proyecto
**Objetivo de la sección (RF-ML7, RF-ML8):**
- Documentar la **Ficha Técnica del Modelo (Model Card)**:
  - **Datos utilizados:** Subconjunto departamental y temporal de SECOP II, fecha de descarga y licencia `CC BY-SA 4.0`.
  - **Uso previsto:** Herramienta de priorización de revisión preventiva para veedurías, periodistas y oficinas de control interno.
  - **Usos NO recomendados:** No debe utilizarse como prueba de incumplimiento, sanción ni calificación automática de corrupción.
  - **Límites conocidos:** Ausencia de documentos anexos del proceso en el dataset tabular, posibles rezagos de registro en SECOP II y cambios normativos o estacionales (ej. Ley de Garantías).